# Atelier 2 — Apprentissage supervisé sur le CO2

L’objectif est de prédire la cible `co2_mixte` exprimée en g/km à partir de caractéristiques techniques de véhicule. Ce problème relève de la régression supervisée car la variable cible est quantitative et continue. Le but n’est pas d’optimiser à tout prix une métrique, mais de vérifier si les variables retenues permettent une bonne généralisation sans fuite entre variantes de la même famille.

## 1. Chargement et préparation du dataset

La donnée est déjà nettoyée. Nous gardons uniquement les variables utiles et nous normalisons les noms de colonnes pour éviter les erreurs d’accès et les fuites de logique. Les colonnes texte sont aussi nettoyées pour garder des familles homogènes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

rnd_state = 42
df = pd.read_csv("../donnees_nettoyees/vehicules_analyse_conso_co2.csv", sep=";", encoding="utf-8-sig")
df.columns = df.columns.str.strip().str.lower()
for col in ["cnit", "lib_mrq_doss", "lib_mod_doss", "energ", "hybride", "typ_boite_nb_rapp"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df["famille"] = df["lib_mrq_doss"].fillna("").astype(str) + " | " + df["lib_mod_doss"].fillna("").astype(str)
print("Dataset chargé :", df.shape)
display(df.head(3))

## 2. Variable cible, variables explicatives et risque de fuite

La cible à prédire est `co2_mixte` exprimée en g/km. Les variables autorisées ici sont techniques et disponibles au moment de la spécification du véhicule : puissance, masse, énergie, hybridation et transmission.

À exclure :
- `conso_mixte` car elle est quasi directement liée à la cible et introduit de la fuite ;
- `cnit` car il s’agit d’un identifiant ;
- les variables de même type ne servant pas à l’interprétation métier.

La colonne `lib_mrq_doss` + `lib_mod_doss` est utilisée uniquement pour constituer le groupe familial, pas comme feature de prédiction.

In [ ]:
target = "co2_mixte"
features = [
    "puiss_admin",
    "puiss_max",
    "masse_ordma_min",
    "energ",
    "hybride",
    "typ_boite_nb_rapp",
]
assert target in df.columns
assert not set(["conso_mixte", "cnit"]).intersection(set(features))
assert set(features).issubset(set(df.columns))

X = df[features].copy()
y = df[target].copy()
groups = df["famille"]
print("Features retenues :", features)
print("Target :", target)

## 3. Diagnostic du split familial

Un split aléatoire simple sur-estime souvent les performances, car des variantes proches d’une même famille peuvent être réparties entre le train et le test. Cela crée une fuite logique. Le diagnostic suivant décrit le comportement du jeu de validation et le risque de domination par quelques familles.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=rnd_state)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Taille train :", X_train.shape)
print("Taille test :", X_test.shape)
print("Nombre de familles train :", groups_train.nunique())
print("Nombre de familles test :", groups_test.nunique())
print("Intersection train/test :", len(set(groups_train.unique()) & set(groups_test.unique())))
print("\nTop familles dans le test :")
print(groups_test.value_counts().head(10).to_string())
print("\nMoyenne CO2 train :", round(y_train.mean(), 2), "; écart-type :", round(y_train.std(), 2))
print("Moyenne CO2 test :", round(y_test.mean(), 2), "; écart-type :", round(y_test.std(), 2))


> **À RÉDIGER (interprétation)** — Questions : (1) Quelles familles dominent le test ? (2) Que révèle la différence de moyenne CO2 ? (3) Quel risque de biais de validation cette composition introduit-elle ? — Chiffres à commenter : voir les statistiques ci-dessus.

## 4. Préprocesseur et pipeline

Le prétraitement est réalisé dans un `Pipeline` avec un `ColumnTransformer` pour garder la même logique de transformation pour tous les modèles. Les variables numériques sont imputées puis standardisées, tandis que les variables catégorielles sont codées via OneHot avec gestion des catégories inconnues.

In [ ]:
numeric_features = ["puiss_admin", "puiss_max", "masse_ordma_min"]
categorical_features = ["energ", "hybride", "typ_boite_nb_rapp"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)
print(preprocessor)

## 5. Baseline et modèles

La baseline est un `DummyRegressor` à la moyenne, que l’on compare à des modèles plus structurés : régression linéaire, arbre de décision, Random Forest et KNN. Cela permet d’évaluer si l’apprentissage exploite réellement les relations entre les variables techniques et le CO2.

In [ ]:
def metric_dict(y_true, y_pred):
    return {
        "r2": r2_score(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
    }

models = {
    "Baseline_mean": DummyRegressor(strategy="mean"),
    "LinearRegression": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(max_depth=6, random_state=rnd_state),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=rnd_state, n_jobs=-1),
    "KNN": KNeighborsRegressor(n_neighbors=7),
}

cv = GroupShuffleSplit(n_splits=20, test_size=0.2, random_state=rnd_state)
rows = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        groups=groups,
        scoring={
            "r2": "r2",
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error",
        },
        n_jobs=1,
    )
    rows.append({
        "model": name,
        "r2_mean": scores["test_r2"].mean(),
        "r2_std": scores["test_r2"].std(),
        "rmse_mean": -scores["test_rmse"].mean(),
        "rmse_std": scores["test_rmse"].std(),
        "mae_mean": -scores["test_mae"].mean(),
        "mae_std": scores["test_mae"].std(),
    })

cv_df = pd.DataFrame(rows).set_index("model")
print("Validation groupée répétée :")
display(cv_df.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, model) in enumerate(models.items(), start=1):
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    scores = cross_validate(
        pipe, X, y, cv=cv, groups=groups, scoring={"r2": "r2"}, n_jobs=1
    )["test_r2"]
    ax.boxplot(scores, positions=[i], patch_artist=True, widths=0.5)
ax.set_xticks(range(1, len(models) + 1))
ax.set_xticklabels(list(models), rotation=30, ha="right")
ax.set_ylabel("R² sur splits groupés")
ax.set_title("Distribution des R² par modèle")
plt.tight_layout()
plt.show()

> **À RÉDIGER (interprétation)** — Questions : (1) Quel modèle est le plus stable sur la validation groupée ? (2) Quel est le niveau de dispersion autour de la moyenne ? (3) Le classement change-t-il par rapport au simple split de diagnostic ? — Chiffres à commenter : voir le tableau et la boîte à moustaches ci-dessus.

## 6. Sensibilité aux familles surreprésentées

On compare maintenant le jeu complet à un jeu plafonné à 50 variantes par famille. Cela permet de vérifier si les performances sont robustes ou si elles dépendent d’une petite poignée de familles très présentes.

In [ ]:
def cap_families(df, max_per_family=50, random_state=42):
    rng = np.random.RandomState(random_state)
    sampled = []
    for fam, fam_df in df.groupby("famille"):
        sample = fam_df.sample(n=min(len(fam_df), max_per_family), random_state=rng.randint(0, 10_000))
        sampled.append(sample)
    return pd.concat(sampled, axis=0)

df_cap = cap_families(df, max_per_family=50, random_state=rnd_state)
X_cap = df_cap[features].copy()
y_cap = df_cap[target].copy()
groups_cap = df_cap["famille"]
cv_cap = GroupShuffleSplit(n_splits=20, test_size=0.2, random_state=rnd_state)
cap_rows = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    scores = cross_validate(pipe, X_cap, y_cap, cv=cv_cap, groups=groups_cap, scoring={"r2": "r2"}, n_jobs=1)
    cap_rows.append({
        "model": name,
        "r2_mean": scores["test_r2"].mean(),
        "r2_std": scores["test_r2"].std(),
    })
cap_df = pd.DataFrame(cap_rows).set_index("model")
print("Jeu plafonné à 50 par famille :")
display(cap_df.round(4))

comparison = cv_df[["r2_mean", "r2_std"]].join(cap_df.rename(columns={"r2_mean": "r2_mean_cap", "r2_std": "r2_std_cap"}))
print("\nComparaison complet / plafonné :")
display(comparison.round(4))

> **À RÉDIGER (interprétation)** — Questions : (1) Le classement change-t-il après plafonnement ? (2) Que dit-on de la robustesse des conclusions ? (3) Quelle part de la variance observée semble liée aux familles extrêmes ? — Chiffres à commenter : voir la comparaison complète/plafonnée ci-dessus.

## 7. Ablation des variables

On compare plusieurs jeux de variables sur la même validation groupée répétée pour mesurer la contribution relative de chaque variable et la dépendance éventuelle à `puiss_admin` et au groupe énergétique.

In [ ]:
ablation_sets = {
    "toutes_variables": features,
    "sans_puiss_admin": [c for c in features if c != "puiss_admin"],
    "sans_masse_ordma_min": [c for c in features if c != "masse_ordma_min"],
    "sans_energ": [c for c in features if c != "energ"],
    "puiss_admin_hybride": ["puiss_admin", "hybride"],
}

def build_preprocessor_for(feature_set):
    num_cols = [c for c in feature_set if c in ["puiss_admin", "puiss_max", "masse_ordma_min"]]
    cat_cols = [c for c in feature_set if c in ["energ", "hybride", "typ_boite_nb_rapp"]]
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
            ("cat", Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
        ]
    )

rows = []
for label, feature_set in ablation_sets.items():
    for model_name in ["LinearRegression", "RandomForest"]:
        X_sel = df[feature_set].copy()
        y_sel = df[target].copy()
        groups_sel = df["famille"]
        pipe = Pipeline(steps=[("preprocessor", build_preprocessor_for(feature_set)), ("model", models[model_name])])
        scores = cross_validate(pipe, X_sel, y_sel, cv=cv, groups=groups_sel, scoring={"r2": "r2"}, n_jobs=1)
        rows.append({
            "jeu": label,
            "modele": model_name,
            "r2_mean": scores["test_r2"].mean(),
            "r2_std": scores["test_r2"].std(),
        })

ablation_df = pd.DataFrame(rows).pivot_table(index="jeu", columns="modele", values="r2_mean")
print("Ablation des variables :")
display(ablation_df.round(4))
print("\nDétail complet :")
display(pd.DataFrame(rows).round(4))

> **À RÉDIGER (interprétation)** — Questions : (1) Que devient la performance sans `puiss_admin` ? (2) Quelle variable apporte le plus à la précision ? (3) Quel rôle joue l’énergie et l’hybridation dans l’explication ? — Chiffres à commenter : voir le tableau ci-dessus.

## 8. Interprétation des variables

On regarde maintenant les relations entre variables et la façon dont le modèle les ordonne. Les coefficients linéaires et l’importance des variables ne sont pas directement comparables, mais ils sont utiles pour questionner la structure apprise par le modèle.

In [ ]:
corr_vars = df[["puiss_admin", "puiss_max", "masse_ordma_min"]].copy()
print("Matrice de corrélation des variables techniques :")
display(corr_vars.corr().round(4))

lin_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", LinearRegression())])
lin_pipe.fit(X_train, y_train)
feature_names = lin_pipe.named_steps["preprocessor"].get_feature_names_out()
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": lin_pipe.named_steps["model"].coef_,
}).sort_values("coef", key=lambda s: s.abs(), ascending=False)
print("\nCoefficients linéaires :")
display(coef_df.head(10).round(4))

rf_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", RandomForestRegressor(n_estimators=300, random_state=rnd_state, n_jobs=-1))])
rf_pipe.fit(X_train, y_train)
imp = pd.DataFrame({
    "feature": rf_pipe.named_steps["preprocessor"].get_feature_names_out(),
    "importance": rf_pipe.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False)
print("\nImportance native du Random Forest :")
display(imp.head(10).round(4))

perm = permutation_importance(rf_pipe, X_test, y_test, n_repeats=5, random_state=rnd_state, n_jobs=-1)
perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_perm": perm.importances_mean,
}).sort_values("importance_perm", ascending=False)
print("\nImportance par permutation :")
display(perm_df.head(10).round(4))

**Avertissement** — Les coefficients linéaires doivent être lus avec précaution car les variables techniques sont fortement corrélées entre elles, notamment `puiss_admin`, `puiss_max` et `masse_ordma_min`. Les colonnes OneHot de `hybride` ont été créées avec gestion des catégories inconnues pour éviter les écarts de codage.

Il n’y a pas ici de preuve de causalité : ce que l’on observe est un diagnostic de dépendance et de structure d’information, pas une explication causaliste du phénomène.

> **À RÉDIGER (interprétation)** — Questions : (1) Quelles variables dominent selon chaque méthode ? (2) Les deux méthodes concordent-elles ? (3) Peut-on lire les coefficients comme des effets causaux ? Pourquoi non ? — Chiffres à commenter : voir les tableaux ci-dessus.

## 9. Erreurs par énergie et par famille

Il est utile de regarder où le modèle échoue le plus : certaines énergies ou certaines familles peuvent concentrer les erreurs les plus fortes. Cela donne un signal métier plus riche que le seul score global.

In [ ]:
best_name = "LinearRegression"
pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", models[best_name])])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
err = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred": pred,
    "abs_err": np.abs(y_test.values - pred),
    "energ": X_test["energ"].values,
    "famille": groups_test.values,
})
print("Erreur absolue moyenne par énergie :")
print(err.groupby("energ")["abs_err"].mean().sort_values(ascending=False).head(10).round(2).to_string())
print("\nErreur absolue moyenne par famille :")
print(err.groupby("famille")["abs_err"].mean().sort_values(ascending=False).head(10).round(2).to_string())

> **À RÉDIGER (interprétation)** — Questions : (1) Où le modèle échoue-t-il le plus ? (2) L’erreur est-elle liée à la rareté ou à la structure du véhicule ? (3) Quelles familles ou énergies méritent un regard particulier ? — Chiffres à commenter : voir les tableaux ci-dessus.

## 10. Synthèse de la chaîne complète

Question métier : peut-on prédire le CO2 d’un véhicule à partir de ses caractéristiques techniques ?

- X : puissance, masse, énergie, hybridation, transmission.
- y : `co2_mixte` en g/km.
- split : train/test et validation groupée par famille pour contrôler la fuite entre variantes proches.
- prétraitement : imputation + standardisation + OneHot.
- baseline : `DummyRegressor(strategy='mean')`.
- modèles : régression linéaire, arbre de décision, RandomForest, KNN.
- métriques : R², RMSE, MAE.
- points de vigilance : familles surreprésentées, colinéarité technique, modèle à comparer sur validation groupée et non sur un split unique.
- interprétation : le modèle doit être compris comme une approximation robuste, pas comme une vérité causaliste.

> **À RÉDIGER (interprétation)** — Questions : (1) Quel modèle retenir pour un décideur ? (2) Quelles sont les limites de l’approche ? (3) Que dirait-on à un décideur en 3 phrases ? — Chiffres à commenter : voir les tableaux et figures ci-dessus.